# 02 ニューラルネット（MLP）— 木に勝てるか。学習のコツと、ぶれの大きさ

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: ニューラルネット（MLP）で予測し、木と比べる回です。過学習と、初期値によるぶれも体験します。
> - **たとえ話**: 小さな部品（ReLU）を重ねて、曲線を作る。重ねすぎると、過去問の丸暗記になる。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-6（ネットのしくみ）・2-7（過学習）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 約3分

## ゴール
- **多層パーセプトロン（MLP）** を PyTorch で学習させ、01 の木（ランダムフォレスト・勾配ブースティング）と比べる。
- **学習のコツ**（標準化・早期終了・過学習）を、学習曲線で理解する。
- **学習シード（初期値）によるぶれ**が、手法の差と同じくらい大きいことを体験する。
- 「特徴量を増やすと必ず良くなる」わけではないことを確かめる。

## 前のNotebookとのつながり
01 では、ランダムフォレストが全特徴量で最良（L=1でMAE 0.785）でした。同じ特徴量を MLP に渡すと、どうなるでしょうか。

## MLP と学習の手順
入力（27個の特徴量）→ 隠れ層（ReLU）→ 出力（発電量）。損失は二乗誤差、最適化は AdamW。
**評価データを使い回さない**ために、2段階で学習します:
1. 学習期間の**最後の2年を「内側の検証」**として取り分け、残りで学習。内側の検証の誤差が**最小のエポック数**を決める（**早期終了**）。
2. 学習期間の**全部**で、そのエポック数だけ学習し直す。評価用（val）は最後に使うだけ。

## キーワード
- **エポック**: 学習データを1周すること。
- **過学習**: 学習データには合うが、新しいデータには合わない。学習損失は下がり続け、検証損失は上がる。
- **早期終了**: 検証損失が最小になった時点で学習をやめる。
- **標準化**: 特徴量を（平均0・標準偏差1）にそろえる。**学習期間の統計だけ**で行う（未来を見ない）。
- **ドロップアウト・重み減衰**: 過学習を抑える仕組み。
- **アンサンブル**: 初期値の違う複数のモデルの予測を平均する。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase2

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 220)

e1, curves = run_phase2.e1()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for a, name in zip(ax, ["小（32）", "中（64,64）", "大（128,128,128）"]):
    tl, el = curves[name]
    a.plot(tl, label="学習の損失"); a.plot(el, label="内側検証の損失")
    a.axvline(int(np.argmin(el)), color="k", ls=":", lw=0.8)
    a.set_title(f"ネット {name}"); a.set_xlabel("エポック"); a.set_ylim(0, 0.2); a.grid(alpha=0.3)
ax[0].set_ylabel("損失（標準化した発電量の二乗誤差）"); ax[0].legend()
plt.tight_layout(); plt.show()
e1

**出力の見方**（点線＝内側検証が最小のエポック）:
- **大きいネット**（128×3層）は、学習の損失が **0.001** まで下がるのに、内側検証の損失は **9エポックで最小（0.059）** のあと、**0.100** まで悪化します。典型的な**過学習**です。
- **小さいネット**（32）は、60エポックまで下がり続け、最後も 0.055 とほぼ最小のまま。**過学習しにくい**。
- 最終的な検証MAE（そのエポック数で学習し直したもの）も、**小 0.832 ＜ 中 0.865 ＜ 大 0.912**。**このデータでは、小さいネットが良い**。ノイズが大きく（ノイズ3%・予報の誤差）、**覚えるべき複雑なパターンが少ない**ためです。
- **標準化なし**（0.854）は、あり（0.865）と**大差ありません**。この特徴量は、もともと値の大きさが近い（0〜30程度）ためです。**スケールがばらばらな特徴量では、標準化は必須**になります（演習で確かめます）。
- ⚠️ 1回ずつの学習なので、**0.01〜0.03の差は初期値のぶれの範囲**かもしれません（E3で調べます）。

In [ ]:
e2, grid = run_phase2.e2()
e2

**出力の見方**（隠れ層4通り×ドロップアウト3通り×重み減衰2通り=24設定を、1つの世界のvalで選び、別の9世界で評価）:
- 選ばれた設定は **隠れ層(32)・ドロップアウト0・重み減衰0.0001**。別の9世界での MAE は **0.827**。
- 既定の(64, 64)は **0.860**。差は 0.033 で、標準誤差（約0.010）の3倍ほど。**小さなネットを選んだ効果は、はっきりあります**。
- 最悪の設定（(16)・ドロップアウト0.3・重み減衰0.01）も 0.847 と、既定より良い。**このデータでは、「ネットを小さく、単純に」すればするほど過学習しにくく、安全**です。
- 上位8設定は、MAE 0.832〜0.861 の狭い範囲に固まっている（下の表）。**設定の細かい違いより、「大きすぎないこと」が大事**。

In [ ]:
grid.head(8)

In [ ]:
e3 = run_phase2.e3()
e3

**出力の見方**（同じ世界・同じデータで、初期値（学習シード）だけを変える）:
- **MLP 1つ**: 10通りの初期値で MAE は **0.852〜0.901**（標準偏差 **0.016**）。**初期値だけで、0.05ものぶれ**があります。木（ランダムフォレスト）のぶれは **0.003** で、約5倍小さい。
- **5つの平均（アンサンブル）**: 平均 **0.845**、標準偏差 **0.006**。**ぶれが約1/3に減り、平均も良くなります**（0.876→0.845）。
- つまり、**1回の学習結果で手法を比べてはいけません**。01 でランダムフォレストとMLPを比べるとき、MLPの順位は**初期値次第**で変わりうる。以降の比較は、**複数の初期値の平均**（アンサンブル）を使います。

In [ ]:
e4 = run_phase2.e4()
piv = e4.pivot(index="手法", columns="リードL", values="MAE")
order = ["予報の直線回帰", "リッジ", "ランダムフォレスト", "勾配ブースティング", "MLP（1つ）", "MLP（5つの平均）",
         "MLP（予報・季節だけ）", "ランダムフォレスト（予報・季節だけ）", "勾配ブースティング（予報・季節だけ）"]
piv.loc[order]

**出力の見方**（選択に使った世界を除く9世界の平均MAE。選んだ設定(32)を使用）:
- **MLP（1つ）**は、全特徴量で L=1: 0.830（ランダムフォレスト 0.783 より +0.047）、L=3: 1.609（+0.102）。**木に負けています**。アンサンブル（5つの平均）で 0.807・1.582 と縮みますが、まだ負けています。L=5 ではほぼ並ぶ（2.629 対 2.623）。
- ところが **MLP（予報・季節だけ）** は、L=1: 0.794（RFと +0.011 で区別できない）、**L=3: 1.477（RFより −0.029、標準誤差 0.013）**、**L=5: 2.570（RFより −0.053、標準誤差 0.009）**。**特徴量を減らしたほうが、MLPは良い**。
- 木も、特徴量を減らすと L=3,5 で良くなります（勾配ブースティング L=5: 2.644→2.598）。L=1 では逆に、特徴量を足したほうが良い（RF 0.834→0.783）。**予報が不正確なリードでは、過去の特徴量は、ノイズとして過学習を招く**。
- **最も低いMAE**: L=1 はランダムフォレスト（全特徴量・0.783）、L=3,5 は MLP（予報・季節だけ）。ただし、MLP（予報・季節だけ）と勾配ブースティング（予報・季節だけ）の差は小さい（L=3: 1.477 対 1.511、L=5: 2.570 対 2.598）。**深層学習が木を明確に超えた、とは言えません**。
- 要するに、この問題の本質は「**予報と季節から、滑らかな関数を学ぶ**」ことで、MLPはその滑らかさに向いている一方、**表データの木と同等**です。

## 試してみよう（演習）
1. **標準化の必要性**: `mlp_fit_predict` に渡す前に、データの `gsr` 列を 1000 倍して（`tr["gsr"] *= 1000` など）、`standardize=True/False` でMAEを比べてください。分かるのは、スケールの違いが学習を壊すことです。
2. **早期終了なし**: `mlp_fit_predict(..., epochs=150)` で、エポック数を固定して学習し、E1 の「大きいネット」の過学習を、検証MAEで確かめてください。
3. **ドロップアウトの効果**: 大きいネット `hidden=(128,128,128)` に `dropout=0.3` を付けて、過学習がどう変わるか見てください。
4. **アンサンブルの数**: `n_seeds` を 1, 3, 5, 10 に変えて、MAEとぶれの変化を見てください。分かるのは、何個で十分かの目安です。

## まとめ
- MLPは、**小さく・単純に**すると良い。大きいネットは**過学習**し、**早期終了**が必要。
- 初期値によるぶれは大きい（MAE 0.852〜0.901）。**アンサンブル**でぶれが1/3に減り、平均も良くなる。**複数の初期値で比べる**。
- **全特徴量**では、MLPは木に負ける（L=1で +0.047）。**予報・季節だけ**にすると、L=3,5 で最も低いMAE。
- **深層学習が表データの木を明確に超えたとは言えない**。同等、またはやや劣る。

次の Phase3 では、**系列モデル（GRU・1D-CNN）** を、生の時系列の窓から学習させます。**隠れた状態（よごれ）を、手作りの特徴量なしで読めるか**が焦点です。